# LAB-D1-02: The Linear Limit

**Purpose:** Observe one linear classifier succeed, then diagnose why the same boundary family cannot represent XOR until a nonlinear hidden transformation changes the coordinates.

**Objectives:** `OBJ-D1-04`, `OBJ-D1-05`, reinforcement of `OBJ-D1-03`  
**Estimated duration:** 45 minutes live; under 20 seconds compute  
**Prerequisites:** `LESSON-D1-03`, `LAB-D1-01`; batch-first NumPy arrays and decision-boundary vocabulary  
**Environment:** CPU only; NumPy, matplotlib, scikit-learn; generated data; no network or download

Workflow: **Observe -> Predict -> Modify -> Run -> Visualize -> Diagnose -> Explain -> Extend**. Start from a clean kernel and run in order. This notebook does not depend on Lab 1 kernel state.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression

SEED = 23
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__} | scikit-learn {sklearn.__version__}")
print("Runtime target: local/Colab CPU; no network or GPU required.")

## Observe: Two Classification Patterns

The first generated dataset has two separated clusters. The second contains four noisy, balanced clusters with alternating XOR targets. Point shape and color both encode class so the plots remain interpretable without relying on color alone.

In [ ]:
X_blobs, y_blobs = make_blobs(
    n_samples=160, centers=[(-1.5, -1.2), (1.4, 1.3)], cluster_std=0.48, random_state=SEED
)
rng = np.random.default_rng(SEED)
xor_centers = np.array([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])
xor_center_labels = np.array([0, 1, 1, 0])
X_xor = np.vstack([rng.normal(center, 0.24, size=(60, 2)) for center in xor_centers])
y_xor = np.repeat(xor_center_labels, 60)

assert X_blobs.shape == (160, 2) and y_blobs.shape == (160,)
assert X_xor.shape == (240, 2) and y_xor.shape == (240,)
assert np.array_equal(np.bincount(y_xor), np.array([120, 120]))
print("Data ready: 160 separable examples and 240 balanced noisy XOR examples.")

In [ ]:
CLASS_STYLES = [(0, "o", "class 0"), (1, "^", "class 1")]


def scatter_classes(ax, X, y, title):
    for label, marker, name in CLASS_STYLES:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], marker=marker, s=42, edgecolor="black", linewidth=0.45, label=name)
    ax.set(xlabel="feature x1", ylabel="feature x2", title=title)
    ax.legend()


def make_mesh(X, padding=0.6, resolution=180):
    x1 = np.linspace(X[:, 0].min() - padding, X[:, 0].max() + padding, resolution)
    x2 = np.linspace(X[:, 1].min() - padding, X[:, 1].max() + padding, resolution)
    xx1, xx2 = np.meshgrid(x1, x2)
    return xx1, xx2, np.column_stack([xx1.ravel(), xx2.ravel()])


def plot_decision_region(ax, X, y, predict_function, title):
    xx1, xx2, grid = make_mesh(X)
    grid_prediction = np.asarray(predict_function(grid)).reshape(xx1.shape)
    ax.contourf(xx1, xx2, grid_prediction, levels=[-0.5, 0.5, 1.5], colors=["#d9e6f2", "#f3ddbd"], alpha=0.65)
    ax.contour(xx1, xx2, grid_prediction, levels=[0.5], colors="black", linewidths=1.8)
    scatter_classes(ax, X, y, title)
    return grid, grid_prediction.ravel()


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
scatter_classes(axes[0], X_blobs, y_blobs, "Observe: separated blobs")
scatter_classes(axes[1], X_xor, y_xor, "Observe: balanced noisy XOR")
plt.tight_layout()
plt.show()

## Predict Before Fitting

Commit to an expected accuracy band and boundary shape for the separated blobs. Then predict an accuracy band and boundary shape for balanced XOR. State whether any expected failure would be caused by representation, parameter choice, optimization time, or another mechanism. Do not fit either model until every field is complete.

In [ ]:
linear_predictions = {
    "blobs_accuracy_band": "",
    "blobs_boundary_shape": "",
    "xor_accuracy_band": "",
    "xor_boundary_shape": "",
    "likely_mechanism": "",
}
assert all(value.strip() for value in linear_predictions.values()), (
    "Prediction checkpoint: complete all five fields in your own words before fitting."
)

## Modify: Fit and Measure One Linear Classifier

Complete the two focused TODOs. Use `LogisticRegression(C=1.0, solver="lbfgs", max_iter=max_iter)` so both datasets use the same explicit estimator settings. `classification_accuracy` should return a scalar fraction, not a percentage string.

In [ ]:
def fit_logistic(X, y, max_iter=1000):
    # TODO: construct the specified estimator, fit it, and return it.
    raise NotImplementedError("TODO: fit and return the specified LogisticRegression model")


def classification_accuracy(y_true, y_pred):
    # TODO: return the fraction of matching class labels.
    raise NotImplementedError("TODO: calculate classification accuracy")

In [ ]:
blob_model = fit_logistic(X_blobs, y_blobs)
blob_predictions = blob_model.predict(X_blobs)
blob_accuracy = classification_accuracy(y_blobs, blob_predictions)
fig, ax = plt.subplots(figsize=(7, 5))
blob_mesh, blob_mesh_predictions = plot_decision_region(ax, X_blobs, y_blobs, blob_model.predict, "Run: logistic regression on separated blobs")
plt.show()
assert np.array_equal(blob_mesh_predictions, blob_model.predict(blob_mesh))
print(f"Separated-blobs accuracy: {blob_accuracy:.3f}")
assert 0.95 <= blob_accuracy <= 1.00, "Check the supplied data and estimator settings."

## Predict Again: XOR and More Iterations

Now that the success case is visible, revisit only your XOR prediction. Will a much larger iteration budget change the family of possible boundaries, or only give the optimizer more opportunity to place a member of the same family? Record one visual observation that could distinguish those explanations.

In [ ]:
xor_recommitment = {"more_iterations": "", "discriminating_visual": ""}
assert all(value.strip() for value in xor_recommitment.values()), (
    "Prediction checkpoint: record the iteration prediction and discriminating visual before the XOR run."
)

In [ ]:
xor_model = fit_logistic(X_xor, y_xor, max_iter=1000)
xor_long_model = fit_logistic(X_xor, y_xor, max_iter=10000)
xor_predictions = xor_model.predict(X_xor)
xor_long_predictions = xor_long_model.predict(X_xor)
xor_accuracy = classification_accuracy(y_xor, xor_predictions)
xor_long_accuracy = classification_accuracy(y_xor, xor_long_predictions)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
plot_decision_region(axes[0], X_xor, y_xor, xor_model.predict, "XOR: standard iteration budget")
plot_decision_region(axes[1], X_xor, y_xor, xor_long_model.predict, "XOR: much larger iteration budget")
plt.tight_layout()
plt.show()
print(f"XOR accuracy: standard={xor_accuracy:.3f}, larger-budget={xor_long_accuracy:.3f}")
assert 0.45 <= xor_accuracy <= 0.60 and 0.45 <= xor_long_accuracy <= 0.60

## Diagnose the Linear Failure

Identify at least two XOR regions that cannot both be placed correctly by the displayed line. Explain why the successful blob result matters to this diagnosis. Distinguish a representation limit from an implementation error and from an optimizer that stopped too early.

In [ ]:
linear_diagnosis = {"inseparable_regions": "", "success_case_value": "", "failure_type": ""}
assert all(value.strip() for value in linear_diagnosis.values()), (
    "Diagnosis checkpoint: complete all three evidence statements before adding a hidden transformation."
)

## Predict: Fixed Two-Unit Hidden Transformation

The next model is not trained here. It uses a supplied `2 -> 2 -> 1` parameter set. Predict what will happen on clean XOR corners and noisy XOR when the hidden activation is ReLU. Then predict what will happen when the activation is replaced by identity while every parameter and data point stays fixed.

Commit to whether adding layer-shaped matrix operations alone is sufficient, or whether the nonlinear transformation is the discriminating change.

In [ ]:
hidden_predictions = {
    "relu_clean_corners": "",
    "relu_noisy_xor": "",
    "identity_same_parameters": "",
    "discriminating_change": "",
}
assert all(value.strip() for value in hidden_predictions.values()), (
    "Prediction checkpoint: complete all hidden-transformation predictions before revealing parameters."
)

In [ ]:
fixed_parameters = {
    "W1": np.array([[1.0, -1.0], [1.0, -1.0]]),
    "b1": np.array([-0.5, -0.5]),
    "W2": np.array([[-4.0], [-4.0]]),
    "b2": np.array([2.0]),
}
fixed_snapshot = {name: value.copy() for name, value in fixed_parameters.items()}
clean_XOR = xor_centers.copy()
clean_XOR_targets = xor_center_labels.copy()
print({name: value.shape for name, value in fixed_parameters.items()})

## Modify: Implement the Fixed Forward Function

Complete the focused TODOs. `two_unit_forward` must return hidden pre-activations `(n, 2)`, hidden activations `(n, 2)`, and probabilities `(n, 1)`. The `hidden_activation` argument lets the same parameter set run with ReLU or identity.

In [ ]:
def relu(values):
    # TODO: apply the elementwise ReLU transformation.
    raise NotImplementedError("TODO: implement ReLU")


def stable_sigmoid(values):
    # TODO: preserve shape and avoid overflow for large magnitudes.
    raise NotImplementedError("TODO: implement a numerically stable sigmoid")


def two_unit_forward(X, parameters, hidden_activation):
    # TODO: compute Z1, H, Z2, and probability using batch-first matrix operations.
    raise NotImplementedError("TODO: implement the fixed 2 -> 2 -> 1 forward pass")


def network_classes(probabilities, threshold=0.5):
    # TODO: threshold the `(n, 1)` probabilities and return shape `(n,)` integers.
    raise NotImplementedError("TODO: convert network probabilities to classes")

In [ ]:
identity = lambda values: values
clean_Z1, clean_hidden, clean_probability = two_unit_forward(clean_XOR, fixed_parameters, relu)
xor_Z1, xor_hidden, xor_probability = two_unit_forward(X_xor, fixed_parameters, relu)
identity_Z1, identity_hidden, identity_probability = two_unit_forward(X_xor, fixed_parameters, identity)
clean_network_accuracy = classification_accuracy(clean_XOR_targets, network_classes(clean_probability))
nonlinear_xor_accuracy = classification_accuracy(y_xor, network_classes(xor_probability))
identity_xor_accuracy = classification_accuracy(y_xor, network_classes(identity_probability))

assert clean_hidden.shape == (4, 2) and clean_probability.shape == (4, 1)
assert xor_hidden.shape == (240, 2) and xor_probability.shape == (240, 1)
assert np.all((xor_probability >= 0.0) & (xor_probability <= 1.0))
print(f"Fixed nonlinear network: clean={clean_network_accuracy:.3f}, noisy={nonlinear_xor_accuracy:.3f}")
print(f"Same parameters with identity activation: noisy={identity_xor_accuracy:.3f}")

## Visualize: Linked Raw and Hidden Coordinates

The same point shapes, class colors, and clean-corner labels appear in both panels. Track point identity rather than treating the hidden plot as a new dataset. The hidden output boundary remains a straight cut; inspect what changed about the coordinates it receives.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter_classes(axes[0], X_xor, y_xor, "Raw XOR coordinates")
for index, point in enumerate(clean_XOR):
    axes[0].annotate(f"C{index}", point, xytext=(5, 5), textcoords="offset points", weight="bold")
    axes[0].scatter(*point, s=150, facecolors="none", edgecolors="black", linewidths=1.8)

scatter_classes(axes[1], xor_hidden, y_xor, "Same examples after ReLU hidden transform")
for index, point in enumerate(clean_hidden):
    axes[1].annotate(f"C{index}", point, xytext=(5, 5), textcoords="offset points", weight="bold")
    axes[1].scatter(*point, s=150, facecolors="none", edgecolors="black", linewidths=1.8)
hidden_x1 = np.linspace(xor_hidden[:, 0].min() - 0.2, xor_hidden[:, 0].max() + 0.2, 160)
hidden_x2 = np.linspace(xor_hidden[:, 1].min() - 0.2, xor_hidden[:, 1].max() + 0.2, 160)
hh1, hh2 = np.meshgrid(hidden_x1, hidden_x2)
hidden_grid = np.column_stack([hh1.ravel(), hh2.ravel()])
hidden_grid_class = network_classes(stable_sigmoid(hidden_grid @ fixed_parameters['W2'] + fixed_parameters['b2'])).reshape(hh1.shape)
axes[1].contour(hh1, hh2, hidden_grid_class, levels=[0.5], colors="black", linewidths=2)
axes[1].set(xlabel="hidden unit h1", ylabel="hidden unit h2")
plt.tight_layout()
plt.show()

In [ ]:
def fixed_predict(X, activation):
    return network_classes(two_unit_forward(X, fixed_parameters, activation)[2])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True, sharey=True)
plot_decision_region(axes[0], X_xor, y_xor, xor_model.predict, "One logistic boundary")
plot_decision_region(axes[1], X_xor, y_xor, lambda X: fixed_predict(X, relu), "Fixed parameters + ReLU")
plot_decision_region(axes[2], X_xor, y_xor, lambda X: fixed_predict(X, identity), "Same parameters + identity")
plt.tight_layout()
plt.show()

## Diagnose and Explain

Use the linked plots and three-region comparison to answer:

1. Why was the blob success an important baseline?
2. What evidence makes XOR a representation failure for one linear boundary?
3. Did the output boundary become complicated, or did the hidden coordinates change?
4. Why did identity activation collapse the layered computation back to a linear family?
5. What evidence would distinguish useful nonlinearity from merely adding parameters?

In [ ]:
explanation = {
    "baseline_value": "",
    "representation_evidence": "",
    "hidden_coordinate_change": "",
    "identity_collapse": "",
    "discriminating_evidence": "",
}
assert all(value.strip() for value in explanation.values()), "Explanation checkpoint: complete all five evidence statements."
assert 0.95 <= blob_accuracy <= 1.00
assert 0.45 <= xor_accuracy <= 0.60
assert clean_network_accuracy == 1.0
assert nonlinear_xor_accuracy > 0.90
assert 0.45 <= identity_xor_accuracy <= 0.60
assert all(np.array_equal(fixed_parameters[name], fixed_snapshot[name]) for name in fixed_parameters)
print("LAB-D1-02 checkpoint passed: success case, linear failure, nonlinear transform, identity control, and parameter parity.")

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Prediction cell stops | A response is empty | Commit a prediction before revealing model evidence |
| Logistic estimator does not fit | The TODO returns an unfitted object or uses incompatible shapes | Fit on `(n, 2)` data and `(n,)` targets, then return the estimator |
| XOR score is unexpectedly high | Data or labels changed, or balance was lost | Restart and rerun the supplied seeded generator unchanged |
| Network matrix error | `W1`, `b1`, `W2`, or `b2` orientation changed | Compare against `(2,2)`, `(2,)`, `(2,1)`, `(1,)` |
| Identity and ReLU use different results | Parameters or data were mutated | Restart; both comparisons must call the same forward function with one activation change |
| Hidden plot seems unrelated | Point identity was not tracked | Follow matching class markers and clean-corner labels across panels |

## Optional Extension

Change only the hidden offset `b1` in a copied parameter dictionary. Predict how the two hidden coordinates and raw-space region will change, then compare under matched axes. Do not mutate `fixed_parameters`.

## Takeaways

- A credible simple model should succeed where its boundary family fits the data.
- More optimization time cannot make one affine threshold bend into XOR regions.
- Hidden nonlinear transformations can make a simple output boundary useful in changed coordinates.
- Layer-shaped affine operations with identity activations still compose to an affine map.
- Raw and hidden plots answer representation questions that aggregate accuracy alone cannot.

Return to the [LAB-D1-02 debrief](../student-guide/day-1-student-guide.md#lab-d1-02---the-linear-limit). Review [LESSON-D1-03](../student-guide/day-1-student-guide.md#lesson-d1-03---from-logistic-regression-to-the-linear-limit) and the prerequisite [LAB-D1-01](LAB-D1-01-neuron-boundary.ipynb) as needed.